# Setup Ollama - Colab

In [65]:
# %pip install colab-xterm
# %load_ext colabxterm

Lanch xtrem terminal in window.

%xterm

Download and run ollama server

curl -sSL https://ollama.ai/install.sh | sh && ollama serve

In [66]:
# !curl http://localhost:11434/api/pull -d '{  "model": "gemma3:12b" }'

In [67]:
# !curl http://localhost:11434/api/pull -d '{  "model": "nomic-embed-text" }'

# Setup tools

In [68]:
# %pip install -qU pandas langchain-ollama langchain-community langchain-text-splitters pypdf

In [69]:
import pandas as pd

from langchain.document_loaders import PyPDFLoader
from langchain_core.vectorstores import InMemoryVectorStore
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_ollama import OllamaLLM, OllamaEmbeddings
from langchain.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough

from pathlib import Path

In [70]:
class CFG:
    model = "gemma3:12b"
    model_embed = "nomic-embed-text"
    rag_file = Path("ustawa_o_sygnalistach.pdf")
    questions_file = "train.csv"


AI_PROMPT = """" 
Jesteś modelem AI, który odpowiada na pytania użytkownika wyłącznie w formie liczbowej.

Użytkownik poda dwa elementy:

Question: pytanie, na które należy odpowiedzieć.
Context: kontekst zawierający dane potrzebne do odpowiedzi.

---

Zasady:

Jeśli w Context znajduje się wartość liczbowa, zwracasz ją jako odpowiedź.
Jeśli Context zawiera przedział liczbowy (np. 10–20), zwracasz maksymalną wartość, chyba że Question wprost wymaga minimum (np. „Jaka jest najmniejsza dopuszczalna kara?” – wtedy zwracasz wartość minimalną).
Jeśli Context zawiera różne liczby, wybierasz tę najbardziej adekwatną do Question.
Jeśli liczba nie jest podana wprost, ale można ją oszacować na podstawie treści (np. „kilka tysięcy” = 2000), zwracasz wartość szacunkową.
Jeśli w Context nie ma liczby, zwracasz „Brak danych”.
Nie podajesz żadnych wyjaśnień, tylko liczbę.

---

Przykłady:

Przykład 1:
Question: Ile osób mieszka w tym mieście?
Context: Populacja miasta wynosi 50 000.
Answer: 50000

Przykład 2:
Question: Jaka jest temperatura wrzenia tej cieczy?
Context: Temperatura wrzenia wynosi 80–100°C.
Answer: 100

Przykład 3:
Question: Jaka jest najmniejsza dopuszczalna kara?
Context: Kara wynosi od 500 do 5000 zł.
Answer: 500

Przykład 4:
Question: Ile lat miał najstarszy uczestnik?
Context: Wiek uczestników wynosił od 25 do 60 lat.
Answer: 60

Przykład 5:
Question: Ile stron ma ta książka?
Context: Brak informacji o liczbie stron.
Answer: Brak danych

---

Question: {question} 
Context: {context}
Answer:
"""

# Load models

In [71]:
try:
    llm = OllamaLLM(model=CFG.model)
    llm_embed = OllamaEmbeddings(model=CFG.model_embed)
except ModuleNotFoundError as e:
    print("Please install ollama first.")
    print(e.msg)
    exit()

# Data preparation

## Load questions

In [72]:
df_qa = pd.read_csv(CFG.questions_file)
df_qa["question"] = df_qa["question"].str.strip()

df_qa.head()

,id,question
0,q1,W ciągu ilu dni od przedstawienia projektu pro...
1,q2,Od ilu zatrudnionych osób podmiot prawny musi ...
2,q3,W jakim terminie (dni) należy potwierdzić sygn...
3,q4,Po ilu miesiącach od upływu terminu kale...
4,q5,Przez ile lat podmiot prawny i organ publiczny...


## Load PDF documents

Documents represent each page in file.

In [73]:
loader = PyPDFLoader(CFG.rag_file)
documents = loader.load()
documents[0]

Document(metadata={'producer': 'Microsoft® Word dla Microsoft 365', 'creator': 'Microsoft® Word dla Microsoft 365', 'creationdate': '2024-06-25T09:38:38+02:00', 'title': 'Ustawa z dnia 14 czerwca 2024 r. o ochronie sygnalistów', 'author': 'RCL', 'moddate': '2024-06-25T09:38:38+02:00', 'source': 'ustawa_o_sygnalistach.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='©Kancelaria Sejmu    s. 1/17 \n      \n \n2024-06-25 \n \n \nDz. U. 2024 poz. 928 \n \n \nUSTAWA  \nz dnia 14 czerwca 2024 r. \no ochronie sygnalistów1), 2) \nRozdział 1 \nPrzepisy ogólne \nArt. 1. Ustawa reguluje: \n1) warunki objęcia ochroną sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n2) środki ochrony sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n3) zasady ustalania wewnętrznej procedury zgłaszania informacji o naruszeniach prawa i podejmowania działań następczych; \n4) zasady zgłaszania informacji o naruszenia

### Split documents into smaller chunks

In [74]:
text_splitter = RecursiveCharacterTextSplitter(chunk_size=2000, chunk_overlap=200)
all_splits = text_splitter.split_documents(documents)

all_splits[0]

Document(metadata={'producer': 'Microsoft® Word dla Microsoft 365', 'creator': 'Microsoft® Word dla Microsoft 365', 'creationdate': '2024-06-25T09:38:38+02:00', 'title': 'Ustawa z dnia 14 czerwca 2024 r. o ochronie sygnalistów', 'author': 'RCL', 'moddate': '2024-06-25T09:38:38+02:00', 'source': 'ustawa_o_sygnalistach.pdf', 'total_pages': 17, 'page': 0, 'page_label': '1'}, page_content='©Kancelaria Sejmu    s. 1/17 \n      \n \n2024-06-25 \n \n \nDz. U. 2024 poz. 928 \n \n \nUSTAWA  \nz dnia 14 czerwca 2024 r. \no ochronie sygnalistów1), 2) \nRozdział 1 \nPrzepisy ogólne \nArt. 1. Ustawa reguluje: \n1) warunki objęcia ochroną sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n2) środki ochrony sygnalistów zgłaszających lub ujawniających publicznie informacje o  naruszeniach prawa; \n3) zasady ustalania wewnętrznej procedury zgłaszania informacji o naruszeniach prawa i podejmowania działań następczych; \n4) zasady zgłaszania informacji o naruszenia

# Vector Database

In [75]:
vector_db = InMemoryVectorStore(embedding=llm_embed)
vector_db.add_documents(all_splits)

['7fdb494a-af48-4ca2-a1c9-b56675835eb9',
 'b856be48-25d6-4e5f-ac22-d5e01845bdbf',
 'df95e326-6fe9-4600-9ef4-53c8362b46f0',
 '9cbc5b55-ab49-4841-a354-fce96a508367',
 '3223e5da-0bc1-4305-af77-314a704e948a',
 'e118e1f0-db14-4c52-87cc-be6e9add808a',
 '0a1f9b37-6b3c-4205-9d7f-026c17dad4c1',
 '69d4f207-68e9-4105-a094-cc0dc60b06f8',
 '85f3361c-1ae5-4d8b-8d80-27927bac508d',
 '1527a652-0776-4601-8c03-2bde355c9af6',
 '88be6d05-dcfb-4072-83aa-6f5d75346cb0',
 '26ea69a8-9077-4713-81cf-dab7039f1e9f',
 '236761ea-3792-488b-b8a9-7602f5e7b659',
 '82c6c581-d6a8-4812-9329-1baa9eb3cf07',
 'a57adcda-428e-4193-bb13-6255be45111a',
 'd3e81c8e-5ebc-446d-b39f-1d81753b8042',
 '4341ee8c-ae74-4f86-a00d-f7164f1795c5',
 'bff793b9-159b-409b-9fa9-d2020796590b',
 '3995c8c9-47d8-4ee1-8448-d4479731206a',
 'ea7feea4-4a2a-4508-a34c-2ff66704ed66',
 '4f4ec7fb-a45c-4b2f-8f8d-cc62886936a3',
 '59e43ace-1540-4a51-96e4-3cabb00fd686',
 '5a34bdf6-02c6-4db8-ba3a-e1286a72dac8',
 'a1117545-db76-41d5-9fbf-4fd63b3b85ab',
 '76df4e78-dce5-

# Prepare prompt

In [76]:
prompt = PromptTemplate(template=AI_PROMPT, input_variables=["context", "question"])

# QA

In [ ]:
def format_docs(docs):
    result = "\n\n".join(doc.page_content for doc in docs)
    return result

In [78]:
qa_chain = (
    {
        "context": vector_db.as_retriever() | format_docs,
        "question": RunnablePassthrough(),
    }
    | prompt
    | llm
    | StrOutputParser()
)

In [80]:
answer = qa_chain.invoke(
    "Ile dni ma organ publiczny na przekazanie zgłoszenia zewnętrznego do właściwego organu? "
)

print(answer)

Brak danych
